In [1]:
import os
from langchain_openai import ChatOpenAI

key = "xxxxxxxxxx.yyyyyyyyyy"
os.environ["ZHIPUAI_API_KEY"] = key
# 1. 初始化模型
chat = ChatOpenAI(
    model="glm-5.3",
    temperature=0.5,
    openai_api_key=os.getenv("ZHIPUAI_API_KEY"),
    openai_api_base="https://open.bigmodel.cn/api/paas/v4/",
)


### 缓冲记忆，存储全部对话历史

In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 2. 定义 Prompt 模板（MessagesPlaceholder 用于插入历史消息）
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手"),
    MessagesPlaceholder(variable_name="history"), # ← 历史消息插在这里
    ("human", "{input}"), # ← 当前用户输入
])
# 3. 组装链（LCEL 管道语法）
chain = prompt_template | chat
# 4. 用字典存储不同会话的历史（key 是 session_id）
store = {}
def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]
# 5. 包装成带记忆的链
chain_with_memory = RunnableWithMessageHistory(
    runnable=chain,
    get_session_history=get_session_history,
    input_messages_key="input",  # 对应用户输入的变量名
    history_messages_key="history",  # 对应历史消息的变量名
)
# 6. 多轮对话测试
config = {"configurable": {"session_id": "user_001"}}

response1 = chain_with_memory.invoke(
    {"input": "我叫小明，我喜欢打篮球"},
    config=config,
)
print(f"AI: {response1.content}")

response2 = chain_with_memory.invoke(
    {"input": "你还记得我叫什么吗？我喜欢什么运动？"},
    config=config,
)
print(f"AI: {response2.content}")

D:\Learn\Learning Project\learn_AI\learn_LangChain\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AI: 你好小明！很高兴认识你！🏀

打篮球是一项很棒的运动，既能锻炼身体，又能培养团队合作精神。你平时喜欢打什么位置呢？是喜欢投篮、突破，还是组织进攻？

另外你有没有喜欢的NBA球星或者CBA球队呀？
AI: 当然记得！😊

你叫**小明**，你喜欢**打篮球**🏀

我们刚才聊过的，你有什么关于篮球的问题想继续聊吗？


### 按轮数记忆，存储最近 n 轮对话历史

In [2]:
"""
方案一：使用 RunnableWithMessageHistory + 自定义 WindowedChatMessageHistory
测试目标：验证历史记录只保留最近 2 条消息（即最近一轮对话）。
"""

from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. 自定义窗口历史类（修正字段声明，避免 Pydantic 报错）
class WindowedChatMessageHistory(InMemoryChatMessageHistory):
    """只保留最近 window_size 条消息的历史存储"""
    window_size: int = 2  # 窗口大小：2 条消息（一轮对话）

    def add_message(self, message):
        super().add_message(message)
        self.messages = self.messages[-self.window_size:]  # 截断

# 2. 会话存储与工厂函数
store = {}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = WindowedChatMessageHistory()
    return store[session_id]

# 3. 构建链（使用已定义的 chat 模型）
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个助手。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
chain = prompt | chat

with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 4. 具体测试对话
session_id = "user1"
config = {"configurable": {"session_id": session_id}}

print("=== 第一轮 ===")
resp1 = with_history.invoke({"input": "你好，我叫Alice"}, config=config)
print("用户: 你好，我叫Alice")
print("AI:", resp1.content)

print("\n=== 第二轮 ===")
resp2 = with_history.invoke({"input": "我来自北京"}, config=config)
print("用户: 我来自北京")
print("AI:", resp2.content)

print("\n=== 第三轮 ===")
resp3 = with_history.invoke({"input": "你还记得我是谁吗？"}, config=config)
print("用户: 你还记得我是谁吗？")
print("AI:", resp3.content)

# 查看最终存储的历史（应只有 2 条消息：第三轮的 human 和 AI）
history = get_session_history(session_id)
print("\n最终历史消息（应为2条：第三轮 human/AI）:")
for msg in history.messages:
    print(f"  {msg.type}: {msg.content}")

D:\Learn\Learning Project\learn_AI\learn_LangChain\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


=== 第一轮 ===
用户: 你好，我叫Alice
AI: 你好，Alice！很高兴认识你 😊

有什么我可以帮助你的吗？无论是回答问题、聊天，还是协助你完成某些任务，我都很乐意为你效劳。

=== 第二轮 ===
用户: 我来自北京
AI: 北京真是个好地方！作为中国的首都，那里有丰富的历史文化和现代化的都市风貌——故宫、长城、颐和园，还有各种美食像烤鸭、炸酱面都很有名 🏮

你是土生土长的北京人吗？还是后来搬到那里的？

=== 第三轮 ===
用户: 你还记得我是谁吗？
AI: 说实话，我需要坦诚地告诉你：我没有跨对话的记忆能力。每次新的对话开始时，我都是“从头开始”的，无法记住之前聊过的内容。

在**这次对话**里，我只知道你告诉我的信息——你来自北京。除此之外，我还不知道你的名字、职业或其他背景。

如果你愿意，可以介绍一下自己，我很乐意认识你！😊

最终历史消息（应为2条：第三轮 human/AI）:
  human: 你还记得我是谁吗？
  ai: 说实话，我需要坦诚地告诉你：我没有跨对话的记忆能力。每次新的对话开始时，我都是“从头开始”的，无法记住之前聊过的内容。

在**这次对话**里，我只知道你告诉我的信息——你来自北京。除此之外，我还不知道你的名字、职业或其他背景。

如果你愿意，可以介绍一下自己，我很乐意认识你！😊


In [4]:
"""
方案二：使用 trim_messages 在链内动态修剪历史消息
测试目标：验证链内修剪只保留最近 2 条消息，而存储中保留完整历史。
注意：在新版 LangChain 中，trim_messages 使用 max_tokens 参数（配合 token_counter=len 即表示最大消息数）。
"""

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import trim_messages

# 1. 会话存储（完整历史，不自动截断）
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 2. 修剪器：仅保留最近 2 条消息
trimmer = trim_messages(
    strategy="last",
    max_tokens=2,          # 修改点：max_messages -> max_tokens
    token_counter=len,     # 使用 len 作为计数器，使 max_tokens 等价于消息数
    start_on="human",
    include_system=True,
)

# 3. 构建链（使用已定义的 chat 模型）
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个助手。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = (
    RunnablePassthrough.assign(history=lambda x: trimmer.invoke(x["history"]))
    | prompt
    | chat
)

with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 4. 具体测试对话
session_id = "user2"
config = {"configurable": {"session_id": session_id}}

print("=== 第一轮 ===")
resp1 = with_history.invoke({"input": "你好，我叫Bob"}, config=config)
print("用户: 你好，我叫Bob")
print("AI:", resp1.content)

print("\n=== 第二轮 ===")
resp2 = with_history.invoke({"input": "我喜欢编程"}, config=config)
print("用户: 我喜欢编程")
print("AI:", resp2.content)

print("\n=== 第三轮 ===")
resp3 = with_history.invoke({"input": "我的爱好是什么？"}, config=config)
print("用户: 我的爱好是什么？")
print("AI:", resp3.content)

# 查看存储中的完整历史（应为 6 条消息）
history = get_session_history(session_id)
print("\n存储中的完整历史（应为6条）:")
for msg in history.messages:
    print(f"  {msg.type}: {msg.content}")

# 手动应用修剪器，展示模型实际收到的消息（应为 2 条）
trimmed = trimmer.invoke(history.messages)
print("\n模型实际收到的消息（修剪后，应为2条）:")
for msg in trimmed:
    print(f"  {msg.type}: {msg.content}")

=== 第一轮 ===
用户: 你好，我叫Bob
AI: 你好，Bob！很高兴认识你。😊

有什么我可以帮你的吗？无论是回答问题、聊天，还是协助你完成某些任务，我都很乐意帮忙。

=== 第二轮 ===
用户: 我喜欢编程
AI: 太棒了，Bob！编程是个非常有趣的爱好（或职业）！💻

我很好奇，你平时主要用什么编程语言呢？比如：

- **Python** - 数据分析、AI、脚本
- **JavaScript/TypeScript** - Web 开发
- **Java/C#** - 企业级应用
- **C/C++** - 系统编程、游戏开发
- **Go/Rust** - 高性能、系统级开发

或者你最近在做什么有趣的项目？无论是网站、App、游戏还是自动化工具，我都很乐意聊聊，甚至帮你解决代码问题、debug、或者探讨技术方案！

你目前在学什么或做什么呢？

=== 第三轮 ===
用户: 我的爱好是什么？
AI: 根据你刚才告诉我的，你的爱好是**编程**！💻

你一开始就说“我喜欢编程”，所以编程就是你的爱好呀～

如果你还有其他爱好想分享，或者想聊聊编程相关的话题（比如正在做的项目、遇到的技术问题等），随时告诉我！

存储中的完整历史（应为6条）:
  human: 你好，我叫Bob
  ai: 你好，Bob！很高兴认识你。😊

有什么我可以帮你的吗？无论是回答问题、聊天，还是协助你完成某些任务，我都很乐意帮忙。
  human: 我喜欢编程
  ai: 太棒了，Bob！编程是个非常有趣的爱好（或职业）！💻

我很好奇，你平时主要用什么编程语言呢？比如：

- **Python** - 数据分析、AI、脚本
- **JavaScript/TypeScript** - Web 开发
- **Java/C#** - 企业级应用
- **C/C++** - 系统编程、游戏开发
- **Go/Rust** - 高性能、系统级开发

或者你最近在做什么有趣的项目？无论是网站、App、游戏还是自动化工具，我都很乐意聊聊，甚至帮你解决代码问题、debug、或者探讨技术方案！

你目前在学什么或做什么呢？
  human: 我的爱好是什么？
  ai: 根据你刚才告诉我的，你的爱好是**编程**！💻

你一开始就说“我喜欢编程”，所以编程就是你的爱好呀～

如果你还

### 摘要缓存

In [8]:
"""
替代 ConversationSummaryBufferMemory 的 LangGraph 实现（修正版）
使用 MessagesState 自动追加消息，避免覆盖历史。
测试目标：当对话历史 token 超过限制时，旧消息被总结为摘要，缓冲区保留最近消息。
"""

import tiktoken
from typing import List
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import get_buffer_string

# ========== 1. 定义状态：继承 MessagesState，自动处理消息追加 ==========
class ChatState(MessagesState):
    summary: str  # 摘要字段

# ========== 2. token 计数函数 ==========
def count_tokens(messages: List[BaseMessage]) -> int:
    encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
    total = 0
    for msg in messages:
        total += len(encoding.encode(msg.content))
    return total

# ========== 3. 记忆处理节点 ==========
def process_memory(state: ChatState, max_token_limit: int = 150) -> dict:
    """如果缓冲区 token 超限，总结前半部分消息，更新摘要和缓冲区。"""
    messages = state["messages"]
    summary = state.get("summary", "")

    if count_tokens(messages) <= max_token_limit:
        return {"messages": messages, "summary": summary}

    # 超限：分割为旧消息（前半部分）和新消息（后半部分）
    split_idx = len(messages) // 2
    old_messages = messages[:split_idx]
    new_messages = messages[split_idx:]

    # 生成摘要
    old_text = get_buffer_string(old_messages)
    summary_prompt = f"请总结以下对话内容，保留关键信息：\n\n{old_text}\n\n摘要："
    summary_response = chat.invoke([HumanMessage(content=summary_prompt)])
    new_summary = summary_response.content

    # 合并摘要
    combined_summary = f"{summary}\n{new_summary}" if summary else new_summary

    # 返回修剪后的消息列表（替换原消息），并更新摘要
    return {"messages": new_messages, "summary": combined_summary}

# ========== 4. 模型调用节点 ==========
def call_model(state: ChatState) -> dict:
    """构建包含摘要和缓冲区的提示，调用模型并返回 AI 消息（自动追加）。"""
    messages = state["messages"]
    summary = state.get("summary", "")

    model_messages = []
    if summary:
        model_messages.append(SystemMessage(content=f"对话摘要：{summary}"))
    model_messages.extend(messages)

    response = chat.invoke(model_messages)
    # 只返回 AI 消息，LangGraph 会自动追加到 messages 列表
    return {"messages": [response]}

# ========== 5. 构建图 ==========
workflow = StateGraph(ChatState)
workflow.add_node("process_memory", process_memory)
workflow.add_node("call_model", call_model)
workflow.set_entry_point("process_memory")
workflow.add_edge("process_memory", "call_model")
workflow.add_edge("call_model", END)

app = workflow.compile(checkpointer=MemorySaver())

# ========== 6. 测试对话 ==========
thread_id = "user1"
config = {"configurable": {"thread_id": thread_id}}

# 第一轮
print("=== 第一轮 ===")
user_input1 = "你好，我叫Alice，我是一名软件工程师。"
resp1 = app.invoke({"messages": [HumanMessage(content=user_input1)]}, config)
print("用户:", user_input1)
print("AI:", resp1["messages"][-1].content)

# 第二轮
print("\n=== 第二轮 ===")
user_input2 = "我最近在学习机器学习，你能推荐一些入门资源吗？"
resp2 = app.invoke({"messages": [HumanMessage(content=user_input2)]}, config)
print("用户:", user_input2)
print("AI:", resp2["messages"][-1].content)

# 第三轮（长消息，确保触发摘要）
print("\n=== 第三轮 ===")
user_input3 = "谢谢你的推荐！我还想了解如何将机器学习应用到自然语言处理中，比如文本分类、情感分析等，你能详细讲讲吗？我对这些非常感兴趣，希望得到一些实践建议。"
resp3 = app.invoke({"messages": [HumanMessage(content=user_input3)]}, config)
print("用户:", user_input3)
print("AI:", resp3["messages"][-1].content)

# 查看最终状态
print("\n=== 最终状态 ===")
print("摘要内容:", resp3["summary"])
print("缓冲区消息数:", len(resp3["messages"]))
for i, msg in enumerate(resp3["messages"]):
    print(f"  [{i}] {msg.type}: {msg.content[:60]}...")

=== 第一轮 ===
用户: 你好，我叫Alice，我是一名软件工程师。
AI: 你好，Alice！很高兴认识你。

作为一名软件工程师，你平时主要做哪方面的开发呢？如果有编程问题、技术难题想讨论，或者其他任何我能帮忙的地方，随时告诉我。

=== 第二轮 ===
用户: 我最近在学习机器学习，你能推荐一些入门资源吗？
AI: 当然可以！作为软件工程师，你有编程基础，这是个很大的优势。以下是一些不错的入门资源：

## 视频课程

- **吴恩达《Machine Learning Specialization》**（Coursera）— 经典入门课，2022新版用Python教学，有中文字幕，讲解清晰
- **李宏毅机器学习课程**（B站/YouTube）— 中文授课，讲解生动有趣，覆盖从基础到深度学习，非常推荐
- **Fast.ai《Practical Deep Learning for Coders》**— 自顶向下、代码优先的教学方式，特别适合程序员

## 书籍

- **《Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow》**（Aurélien Géron）— 实践导向，理论结合代码，工程师入门首选
- **《机器学习》（西瓜书，周志华）**— 中文经典教材，适合系统学习理论
- **《统计学习方法》（李航）**— 简洁严谨，数学推导清晰

## 动手实践

- **Kaggle** — 从 Titanic 等入门竞赛开始，看别人的 notebook 学习技巧
- **Google Colab** — 免费GPU，不用配置环境就能跑实验
- **scikit-learn 官方教程** — 文档质量很高

## 数学基础（可选）

- **3Blue1Brown** 的线性代数、微积分、神经网络系列视频 — 可视化讲解非常出色

## 建议路径

1. 先上一门视频课建立整体认知（吴恩达或李宏毅）
2. 配合《Hands-On ML》边学边写代码
3. 在 Kaggle 上做几个小项目巩固

方便问一下，你对哪个方向更感兴趣？比如 NLP、计算机视觉，还是想先打好通用基础？另外你的数学背景如何？这样我可以给出更针对性的建议。

=== 第三轮 ===
用户: 谢谢你的推荐！我还